# onto-canon6 Donor Profile Loading

This notebook proves that `onto-canon6` can load real donor profiles and packs
from `onto-canon5` without importing the old runtime.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEARCH_ROOTS = [
    Path.cwd().resolve(),
    Path('~/projects/onto-canon6').expanduser(),
]

PROJECT_ROOT = None
for start in SEARCH_ROOTS:
    candidate = start
    while True:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    if PROJECT_ROOT is not None:
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate onto-canon6 repo root from notebook cwd or ~/projects/onto-canon6"
    )

for candidate_path in (
    PROJECT_ROOT / "src",
    PROJECT_ROOT.parent / "llm_client",
    Path('~/projects/llm_client').expanduser(),
):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from pprint import pprint



from onto_canon6.ontology_runtime import (  # noqa: E402
    decide_unknown_item,
    donor_ontology_packs_root,
    donor_profiles_root,
    load_ontology_pack,
    load_profile,
)

print(PROJECT_ROOT)
print('profiles root:', donor_profiles_root())
print('packs root:', donor_ontology_packs_root())


/home/brian/projects/onto-canon6
profiles root: /home/brian/projects/onto-canon5/profiles
packs root: /home/brian/projects/onto-canon5/ontology_packs


In [2]:
PROFILE_KEYS = [
    ("default", "1.0.0"),
    ("dodaf", "0.1.0"),
    ("psyop_seed", "0.1.0"),
]

rows = []
for profile_id, profile_version in PROFILE_KEYS:
    profile = load_profile(profile_id, profile_version)
    rows.append(
        {
            'profile': f'{profile.profile_id}@{profile.profile_version}',
            'mode': profile.ontology_policy.mode,
            'proposal_policy': profile.ontology_policy.proposal_policy,
            'pack_ref': profile.pack_ref.model_dump() if profile.pack_ref else None,
            'allowed_predicate_count': len(profile.allowed_predicates or []),
            'predicate_rule_count': len(profile.predicate_rules),
            'unknown_predicate_severity': profile.severity_by_code.get('oc:profile_unknown_predicate'),
        }
    )

pprint(rows)


[{'allowed_predicate_count': 0,
  'mode': 'open',
  'pack_ref': None,
  'predicate_rule_count': 0,
  'profile': 'default@1.0.0',
  'proposal_policy': 'reject',
  'unknown_predicate_severity': 'soft'},
 {'allowed_predicate_count': 9,
  'mode': 'closed',
  'pack_ref': None,
  'predicate_rule_count': 9,
  'profile': 'dodaf@0.1.0',
  'proposal_policy': 'reject',
  'unknown_predicate_severity': 'hard'},
 {'allowed_predicate_count': 14,
  'mode': 'mixed',
  'pack_ref': {'pack_id': 'onto_canon_psyop_seed', 'pack_version': '0.1.0'},
  'predicate_rule_count': 14,
  'profile': 'psyop_seed@0.1.0',
  'proposal_policy': 'allow',
  'unknown_predicate_severity': 'soft'}]


In [3]:
psyop_pack = load_ontology_pack('onto_canon_psyop_seed', '0.1.0')
pprint(
    {
        'pack_ref': psyop_pack.pack_ref.model_dump(),
        'predicate_count': len(psyop_pack.predicate_ids),
        'predicate_rule_count': len(psyop_pack.predicate_rules),
        'sample_predicates': sorted(list(psyop_pack.predicate_ids))[:5],
        'sample_type_parents': list(sorted(psyop_pack.type_parents.items()))[:5],
    }
)


{'pack_ref': {'pack_id': 'onto_canon_psyop_seed', 'pack_version': '0.1.0'},
 'predicate_count': 14,
 'predicate_rule_count': 14,
 'sample_predicates': ['oc:belongs_to_organization',
                       'oc:create_organizational_unit',
                       'oc:criticize_change',
                       'oc:describe_dissatisfaction',
                       'oc:express_concern'],
 'sample_type_parents': [('oc:community', ('oc:organization',)),
                         ('oc:government_organization', ('oc:organization',)),
                         ('oc:information_content', ('oc:entity',)),
                         ('oc:military_operation', ('oc:entity',)),
                         ('oc:military_organization',
                          ('oc:government_organization',))]}


In [4]:
for profile_id, profile_version in PROFILE_KEYS:
    profile = load_profile(profile_id, profile_version)
    decision = decide_unknown_item(
        policy=profile.ontology_policy,
        kind='predicate',
        value='oc:unknown_predicate_demo',
    )
    print(profile_id, decision.action, decision.reason)


default allow default ontology mode 'open' resolved action 'allow' for predicate
dodaf reject default ontology mode 'closed' resolved action 'reject' for predicate
psyop_seed propose default ontology mode 'mixed' resolved action 'propose' for predicate
